# Day 1: notebook-local vLLM serving
Run in a Colab GPU runtime. Upload a committed source archive and enter its SHA-256. No public tunnel or paid API is used. GPU allocation and compatibility are observed here, not assumed.

In [ ]:
import json, platform, subprocess, sys, time, hashlib, pathlib, zipfile, os, signal
print(json.dumps({"python": sys.version.split()[0], "platform": platform.platform(), "nvidia_smi": subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()}, indent=2))


## Upload a pinned source archive
On the Mac, run the commands in notebooks/README.md. Set EXPECTED_SHA256 before uploading.

In [ ]:
EXPECTED_SHA256 = ""  # paste the 64-character shasum for jev-source.zip
assert len(EXPECTED_SHA256) == 64, "Set EXPECTED_SHA256 from the local committed archive"
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1
archive_name, archive_bytes = next(iter(uploaded.items()))
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_SHA256
with zipfile.ZipFile(archive_name) as z:
    assert all(not pathlib.PurePosixPath(p).is_absolute() and ".." not in pathlib.PurePosixPath(p).parts for p in z.namelist())
    z.extractall("/content/jev-source")
source_dir = pathlib.Path("/content/jev-source")
assert (source_dir / "pyproject.toml").exists()


## Install the same gateway source and a pinned GPU runtime
The vLLM wheel must match the actual Colab GPU and CUDA driver; inspect the preflight before running. This may use substantial notebook disk and GPU quota.

In [ ]:
VLLM_VERSION = "0.18.0"
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", str(source_dir), f"vllm=={VLLM_VERSION}"], check=True)
import jev_gateway, vllm
print({"gateway": jev_gateway.__version__, "vllm": vllm.__version__, "source_sha256": EXPECTED_SHA256})


## Start a loopback-only model and check readiness
The selected small instruction model is a candidate. If the allocated GPU is insufficient, stop and preserve the failure report.

In [ ]:
import urllib.request, shutil
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
log_path = pathlib.Path("/content/vllm-day1.log")
log_file = log_path.open("w")
server = subprocess.Popen([shutil.which("vllm"), "serve", MODEL, "--host", "127.0.0.1", "--port", "8000", "--max-model-len", "2048", "--gpu-memory-utilization", "0.75"], stdout=log_file, stderr=subprocess.STDOUT)
for attempt in range(120):
    if server.poll() is not None: raise RuntimeError(f"vLLM exited {server.returncode}; inspect {log_path}")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as r:
            print(r.read().decode()[:2000]); break
    except Exception: time.sleep(2)
else: raise TimeoutError("vLLM readiness exceeded 240 seconds")


## Two bounded notebook-local requests and export

In [ ]:
import httpx
results = []
with httpx.Client(timeout=60) as client:
    for prompt in ("Reply with one word: hello", "What is two plus two? Reply with one digit."):
        r = client.post("http://127.0.0.1:8000/v1/chat/completions", json={"model": MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": 16, "temperature": 0})
        results.append({"status": r.status_code, "response": r.json()})
report = {"source_sha256": EXPECTED_SHA256, "vllm_version": vllm.__version__, "model": MODEL, "results": results}
pathlib.Path("/content/jev-day1-results.json").write_text(json.dumps(report, indent=2))
files.download("/content/jev-day1-results.json")


## Cleanup

In [ ]:
server.terminate()
try: server.wait(timeout=15)
except subprocess.TimeoutExpired: server.kill(); server.wait()
log_file.close()
print("vLLM stopped", server.returncode)
